In [2]:
!pip install requests

In [ ]:
#Cut 1 Spectral Download
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from astropy.table import Table

# ----------------------------
# User settings
# ----------------------------
ID_FILE = "apogee_ids_2M_Cut_1.txt"
ALLSTAR_FILE = "allStar-dr17-synspec_rev1.fits"
OUTPUT_DIR = "apogee_apStar_dr17"
SLEEP_BETWEEN = 0.1

# ----------------------------
# Load APOGEE IDs from text file
# ----------------------------
with open(ID_FILE, "r") as f:
    ids = [line.strip() for line in f if line.strip()]

# remove duplicates while preserving order
ids = list(dict.fromkeys(ids))

print(f"Loaded {len(ids)} APOGEE IDs")

# ----------------------------
# Read allStar and recover FIELD/TELESCOPE
# ----------------------------
tab = Table.read(ALLSTAR_FILE, hdu=1)

tab_ids = np.array([str(x).strip() for x in tab["APOGEE_ID"]])

want = set(ids)
rows = []

for row, apid in zip(tab, tab_ids):
    if apid in want:
        rows.append({
            "APOGEE_ID": apid,
            "FIELD": str(row["FIELD"]).strip(),
            "TELESCOPE": str(row["TELESCOPE"]).strip(),
        })

meta = pd.DataFrame(rows).drop_duplicates(subset=["APOGEE_ID"])
meta = meta.set_index("APOGEE_ID").reindex(ids).reset_index()

missing = meta["FIELD"].isna() | meta["TELESCOPE"].isna()
if missing.any():
    print("\nIDs not found in allStar:")
    for apid in meta.loc[missing, "APOGEE_ID"]:
        print(apid)

meta = meta.loc[~missing].copy()
print(f"Found FIELD/TELESCOPE for {len(meta)} IDs")

display(meta.head())

# ----------------------------
# Make output directory
# ----------------------------
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

# ----------------------------
# DR17 apStar base URL
# ----------------------------
base = "https://data.sdss.org/sas/dr17/apogee/spectro/redux/dr17/stars"

# ----------------------------
# Download helper
# ----------------------------
def download_file(url, outpath):
    if outpath.exists():
        print(f"Already exists: {outpath.name}")
        return True

    try:
        r = requests.get(url, stream=True, timeout=60)
        if r.status_code != 200:
            print(f"Failed ({r.status_code}): {url}")
            return False

        with open(outpath, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

        print(f"Saved: {outpath.name}")
        return True

    except Exception as e:
        print(f"Error downloading {url}: {e}")
        return False

# ----------------------------
# Download all spectra
# ----------------------------
success = 0
failed = 0

for _, row in meta.iterrows():
    apogee_id = row["APOGEE_ID"]
    field = row["FIELD"]
    telescope = row["TELESCOPE"]

    filename = f"apStar-dr17-{apogee_id}.fits"
    url = f"{base}/{telescope}/{field}/{filename}"
    outpath = output_dir / filename

    ok = download_file(url, outpath)
    if ok:
        success += 1
    else:
        failed += 1

    time.sleep(SLEEP_BETWEEN)

print(f"\nDone. Success: {success}, Failed: {failed}")

Loaded 111 APOGEE IDs
Found FIELD/TELESCOPE for 111 IDs


,APOGEE_ID,FIELD,TELESCOPE
0,2M08263810-1806097,240+12,lco25m
1,2M08264174-1805195,240+12,lco25m
2,2M08264810-1810234,240+12,lco25m
3,2M08264981-1808416,240+12,lco25m
4,2M08265265-1806186,240+12,lco25m


Failed (404): https://data.sdss.org/sas/dr17/apogee/spectro/redux/dr17/stars/lco25m/240+12/apStar-dr17-2M08263810-1806097.fits
Failed (404): https://data.sdss.org/sas/dr17/apogee/spectro/redux/dr17/stars/lco25m/240+12/apStar-dr17-2M08264174-1805195.fits
Failed (404): https://data.sdss.org/sas/dr17/apogee/spectro/redux/dr17/stars/lco25m/240+12/apStar-dr17-2M08264810-1810234.fits
Failed (404): https://data.sdss.org/sas/dr17/apogee/spectro/redux/dr17/stars/lco25m/240+12/apStar-dr17-2M08264981-1808416.fits
Failed (404): https://data.sdss.org/sas/dr17/apogee/spectro/redux/dr17/stars/lco25m/240+12/apStar-dr17-2M08265265-1806186.fits
Failed (404): https://data.sdss.org/sas/dr17/apogee/spectro/redux/dr17/stars/lco25m/240+12/apStar-dr17-2M08265861-1808089.fits
Failed (404): https://data.sdss.org/sas/dr17/apogee/spectro/redux/dr17/stars/lco25m/240+12/apStar-dr17-2M08265916-1813079.fits
Failed (404): https://data.sdss.org/sas/dr17/apogee/spectro/redux/dr17/stars/lco25m/240+12/apStar-dr17-2M082701

In [10]:
df2 = pd.read_csv("apogee_stars_RA_Dec_cut2_smaller.csv", sep=',')
df2['APOGEE_ID']
df2['APOGEE_ID'].to_csv('apogee_ids_2M_Cut_2.txt', index=False, header=False, sep='\t')

In [11]:
#Cut 2 Spectral Download
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from astropy.table import Table

# ----------------------------
# User settings
# ----------------------------
ID_FILE = "apogee_ids_2M_Cut_2.txt"
ALLSTAR_FILE = "allStar-dr17-synspec_rev1.fits"
OUTPUT_DIR = "apogee_apStar_dr17_cut2"
SLEEP_BETWEEN = 0.1

# ----------------------------
# Load APOGEE IDs from text file
# ----------------------------
with open(ID_FILE, "r") as f:
    ids = [line.strip() for line in f if line.strip()]

# remove duplicates while preserving order
ids = list(dict.fromkeys(ids))

print(f"Loaded {len(ids)} APOGEE IDs")

# ----------------------------
# Read allStar and recover FIELD/TELESCOPE
# ----------------------------
tab = Table.read(ALLSTAR_FILE, hdu=1)

tab_ids = np.array([str(x).strip() for x in tab["APOGEE_ID"]])

want = set(ids)
rows = []

for row, apid in zip(tab, tab_ids):
    if apid in want:
        rows.append({
            "APOGEE_ID": apid,
            "FIELD": str(row["FIELD"]).strip(),
            "TELESCOPE": str(row["TELESCOPE"]).strip(),
        })

meta = pd.DataFrame(rows).drop_duplicates(subset=["APOGEE_ID"])
meta = meta.set_index("APOGEE_ID").reindex(ids).reset_index()

missing = meta["FIELD"].isna() | meta["TELESCOPE"].isna()
if missing.any():
    print("\nIDs not found in allStar:")
    for apid in meta.loc[missing, "APOGEE_ID"]:
        print(apid)

meta = meta.loc[~missing].copy()
print(f"Found FIELD/TELESCOPE for {len(meta)} IDs")

display(meta.head())

# ----------------------------
# Make output directory
# ----------------------------
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

# ----------------------------
# DR17 apStar base URL
# ----------------------------
base = "https://data.sdss.org/sas/dr17/apogee/spectro/redux/dr17/stars"

# ----------------------------
# Download helper
# ----------------------------
def download_file(url, outpath):
    if outpath.exists():
        print(f"Already exists: {outpath.name}")
        return True

    try:
        r = requests.get(url, stream=True, timeout=60)
        if r.status_code != 200:
            print(f"Failed ({r.status_code}): {url}")
            return False

        with open(outpath, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

        print(f"Saved: {outpath.name}")
        return True

    except Exception as e:
        print(f"Error downloading {url}: {e}")
        return False

# ----------------------------
# Download all spectra
# ----------------------------
success = 0
failed = 0

for _, row in meta.iterrows():
    apogee_id = row["APOGEE_ID"]
    field = row["FIELD"]
    telescope = row["TELESCOPE"]

    filename = f"apStar-dr17-{apogee_id}.fits"
    url = f"{base}/{telescope}/{field}/{filename}"
    outpath = output_dir / filename

    ok = download_file(url, outpath)
    if ok:
        success += 1
    else:
        failed += 1

    time.sleep(SLEEP_BETWEEN)

print(f"\nDone. Success: {success}, Failed: {failed}")

Loaded 3413 APOGEE IDs
Found FIELD/TELESCOPE for 3413 IDs


,APOGEE_ID,FIELD,TELESCOPE
0,2M16200012+2733344,046+45_MGA,apo25m
1,2M16200058+2638119,043+44_MGA,apo25m
2,2M16200096+3029038,049+46_MGA,apo25m
3,2M16200130+3054529,052+44_MGA,apo25m
4,2M16200198+2825014,048+44_MGA,apo25m


Saved: apStar-dr17-2M16200012+2733344.fits
Saved: apStar-dr17-2M16200058+2638119.fits
Saved: apStar-dr17-2M16200096+3029038.fits
Saved: apStar-dr17-2M16200130+3054529.fits
Saved: apStar-dr17-2M16200198+2825014.fits
Saved: apStar-dr17-2M16200223+3323551.fits
Saved: apStar-dr17-2M16200263+3300390.fits
Saved: apStar-dr17-2M16200295+2810183.fits
Saved: apStar-dr17-2M16200351+3111508.fits
Saved: apStar-dr17-2M16200366+2757487.fits
Saved: apStar-dr17-2M16200528+3240432.fits
Saved: apStar-dr17-2M16200550+3313093.fits
Saved: apStar-dr17-2M16200675+2811476.fits
Saved: apStar-dr17-2M16200695+2625173.fits
Saved: apStar-dr17-2M16200725+2804191.fits
Saved: apStar-dr17-2M16200818+3115377.fits
Saved: apStar-dr17-2M16201057+3130398.fits
Saved: apStar-dr17-2M16201156+2900436.fits
Saved: apStar-dr17-2M16201263+3226245.fits
Saved: apStar-dr17-2M16201277+3057381.fits
Saved: apStar-dr17-2M16201277+2545026.fits
Saved: apStar-dr17-2M16201504+2541172.fits
Saved: apStar-dr17-2M16201556+3104204.fits
Saved: apSt

In [1]:
#Spectral Extraction/Plotting (Cut 1)
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

input_dir = Path("/Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_apStar_dr17")
output_dir = Path("/Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots")
output_dir.mkdir(parents=True, exist_ok=True)

files = sorted(input_dir.glob("*.fits"))
print(f"Found {len(files)} FITS files")


for fname in files:
    try:
        with fits.open(fname) as hdul:
            flux = hdul[1].data
            hdr = hdul[1].header

            crval1 = hdr["CRVAL1"]
            cdelt1 = hdr["CDELT1"]
            naxis1 = hdr["NAXIS1"]

            loglam = crval1 + np.arange(naxis1) * cdelt1
            wave = 10**loglam  # Angstrom

            if flux.ndim == 2:
                flux = flux[0]

        plt.figure(figsize=(14, 4))
        plt.plot(wave / 1e4, flux, lw=0.7)
        plt.xlabel("Wavelength [micron]")
        plt.ylabel("Flux")
        plt.title(fname.stem)
        plt.tight_layout()

        apogee_id = fname.stem.replace("apStar-dr17-", "")
        outpath = output_dir / f"{apogee_id}.png"
        plt.savefig(outpath, dpi=200, bbox_inches="tight")
        plt.close()

        print(f"Saved {outpath}")

    except Exception as e:
        print(f"Failed on {fname.name}: {e}")

Found 79 FITS files
Saved /Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots/2M08313774-2044159.png
Saved /Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots/2M08313948-2049340.png
Saved /Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots/2M08314033-2045418.png
Saved /Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots/2M08314056-2051062.png
Saved /Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots/2M08314088-2058127.png
Saved /Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots/2M08314312-2052081.png
Saved /Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots/2M08314570-2035331.png
Saved /Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots/2M08315521-2059494.png
Saved /Users/sarahstamer/Desktop/Grad/2025-2

In [1]:
#Spectral Extraction/Plotting (Cut 2)
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

input_dir = Path("/Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_apStar_dr17_cut2")
output_dir = Path("/Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots_cut2")
output_dir.mkdir(parents=True, exist_ok=True)

files = sorted(input_dir.glob("*.fits"))
print(f"Found {len(files)} FITS files")


for fname in files:
    try:
        with fits.open(fname) as hdul:
            flux = hdul[1].data
            hdr = hdul[1].header

            crval1 = hdr["CRVAL1"]
            cdelt1 = hdr["CDELT1"]
            naxis1 = hdr["NAXIS1"]

            loglam = crval1 + np.arange(naxis1) * cdelt1
            wave = 10**loglam  # Angstrom

            if flux.ndim == 2:
                flux = flux[0]

        plt.figure(figsize=(14, 4))
        plt.plot(wave / 1e4, flux, lw=0.7)
        plt.xlabel("Wavelength [micron]")
        plt.ylabel("Flux")
        plt.title(fname.stem)
        plt.tight_layout()

        apogee_id = fname.stem.replace("apStar-dr17-", "")
        outpath = output_dir / f"{apogee_id}.png"
        plt.savefig(outpath, dpi=200, bbox_inches="tight")
        plt.close()

        print(f"Saved {outpath}")

    except Exception as e:
        print(f"Failed on {fname.name}: {e}")

Found 3365 FITS files
Saved /Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots_cut2/2M16200012+2733344.png
Saved /Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots_cut2/2M16200058+2638119.png
Saved /Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots_cut2/2M16200096+3029038.png
Saved /Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots_cut2/2M16200130+3054529.png
Saved /Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots_cut2/2M16200198+2825014.png
Saved /Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots_cut2/2M16200223+3323551.png
Saved /Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots_cut2/2M16200263+3300390.png
Saved /Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots_cut2/2M16200295+2810183.png
Sa

Saved /Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots_cut2/2M16531283+2705098.png
Saved /Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots_cut2/2M16531290+2557170.png
Saved /Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots_cut2/2M16531365+3414261.png
Saved /Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots_cut2/2M16531497+2802538.png
Saved /Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots_cut2/2M16531575+3318362.png
Saved /Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots_cut2/2M16531586+3425277.png
Saved /Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots_cut2/2M16531697+2610289.png
Saved /Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots_cut2/2M16531709+2509097.png
Saved /Users/sarahstamer

Saved /Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots_cut2/2M16551984+3048183.png
Saved /Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots_cut2/2M16552070+3231489.png
Saved /Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots_cut2/2M16552222+2852313.png
Saved /Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots_cut2/2M16552261+2502395.png
Saved /Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots_cut2/2M16552301+3358099.png
Saved /Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots_cut2/2M16552356+3258504.png
Saved /Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots_cut2/2M16552382+3234413.png
Saved /Users/sarahstamer/Desktop/Grad/2025-2026/ASTR_526/Project/APOGEE/apogee_spectrum_plots_cut2/2M16552391+2811151.png
Saved /Users/sarahstamer